# Phase 1 Stable Baseline — SentinelFlow

Single end-to-end notebook that rebuilds the Phase 1 mule-account classifier from the **PS2 golden-truth report** (`ps2_mule_account_detection_report.pdf`), with every previously identified mistake corrected in *new* files only.

## What this notebook is
- One sequential pipeline: load → **leakage retest with numbers** → compact features → fold-safe encodings → calibrated GBDT ensemble → cost threshold → chronological check.
- Helpers are copied from `Phase_1/` (`build_row_stats`, fold-safe target encoding, Isolation Forest, isotonic + cost search). Nothing from `Description.xlsx` is used here; that file is reserved for Phase 2 fine-tuning.

## Fixes versus the original Phase 1 notebooks
| Issue in Phase 1 | Stable baseline |
|---|---|
| Leaky columns dropped *before* the audit, so they could not be retested | Audit runs on the raw table first; decisions are printed with rates, support, Cohen's d, and a holdout ablation |
| `min fraud rate < 0.001` flagged almost every rare-event category | Purity rule requires **support ≥ 20** and **fraud rate ≥ 0.99**, or a numeric proxy (\|corr\| ≥ 0.5 and \|d\| ≥ 2) |
| `F3889` / `F3891` dropped even though the bank list and PS2 say salvage them | `F3889` mapped to a numeric recency lag; `F3891` kept as fold-safe occupation encoding |
| SMOTE still in steps 4–6 after the report banned it | `scale_pos_weight` only |
| `elapsed_days` timeline anchor (step 6 showed it is unnecessary) | Cyclical `F3888` sin/cos only |
| Global PCA / KMeans fitted on all rows | Dropped; they were not in the PS2 105-feature tracks and leak across folds |
| `DATA_PATH = DataSet.csv` broke when the cwd was `Phase_1/` | Paths resolve from this folder to the repo root |

## Leakage policy after retest (preview; cells below print the evidence)
- **Exclude:** `F3912` (numeric target proxy), `F2230` (month codes that perfectly separate fraud).
- **Keep / salvage:** `F3886` (account type), `F3889` (recency buckets → lag), `F3891` (occupation), `F3892` (gender). These were false positives of the old heuristic.

## 0. Setup

Imports, constants, and repo-root paths. `Description.xlsx` is detected and then ignored on purpose.

In [ ]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.calibration import IsotonicRegression
from sklearn.ensemble import IsolationForest
from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import RobustScaler

warnings.filterwarnings("ignore", message="Could not infer format")
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

RANDOM_STATE = 42
TARGET_COL = "F3924"
ID_COL = "Unnamed: 0"

HERE = Path.cwd().resolve()
ROOT = HERE if (HERE / "DataSet.csv").exists() else HERE.parent
DATA_PATH = ROOT / "DataSet.csv"
DESC_PATH = ROOT / "Description.xlsx"
OUT_DIR = HERE if HERE.name == "Phase_1_stable" else HERE / "Phase_1_stable"
OUT_DIR.mkdir(exist_ok=True)

# Bank-listed features from Topic.pdf / Problem_statement.md (not from Description.xlsx)
BANK_FEATURES = [
    "F115", "F321", "F527", "F531", "F670", "F1692", "F2082", "F2122",
    "F2582", "F2678", "F2737", "F2956", "F3043", "F3836", "F3887",
    "F3889", "F3891", "F3894",
]

# Previously accused columns. Retested below; this list is NOT dropped blindly.
PREVIOUSLY_ACCUSED = ["F3912", "F2230", "F3886", "F3889", "F3891", "F3892"]

PLACEHOLDER_VALUES = {-99999999, 99999999, -9999999, 9999999, -999999, 999999, -9999, 9999}
LARGE_ABS_THRESHOLD = 1e7
PLACEHOLDER_MIN_FRAC = 0.002

TOP_MI = 25
TOP_GAP = 25
GAP_MIN = 0.20
LOW_CARD_MAX_UNIQUE = 12
TARGET_ENCODING_SMOOTHING = 20.0
TEMPORAL_PARSE_MIN_FRAC = 0.7
ANOMALY_N_ESTIMATORS = 200
BLEND_WEIGHTS = (0.6, 0.4)
COST_FN_RATIO = 5.0
COST_FP_BASE = 1.0
N_SPLITS = 5
MI_CANDIDATE_CAP = 800
PURITY_MIN_SUPPORT = 20
PURITY_MIN_RATE = 0.99
PROXY_MIN_ABS_CORR = 0.50
PROXY_MIN_ABS_D = 2.0

# F3889 recency codes observed in the raw table (not dates).
F3889_LAG_MAP = {
    "L7D": 7,
    "L14D": 14,
    "L31D": 31,
    "L90D": 90,
    "L180D": 180,
    "L365D": 365,
    "G365D": 400,
}

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 80)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")

try:
    import xgboost as xgb
except ImportError:
    xgb = None
    print("xgboost not installed. pip install xgboost")

try:
    import lightgbm as lgb
except ImportError:
    lgb = None
    print("lightgbm not installed. pip install lightgbm")

print("ROOT:", ROOT)
print("DATA_PATH exists:", DATA_PATH.exists())
print("Description.xlsx exists:", DESC_PATH.exists(), "— reserved for Phase 2 fine-tuning, unused here.")

## 1. Load and integrity checks

Copied from `eda.ipynb` / Phase 1 load cells. Object columns are **kept** (Phase 1 often coerced everything to numeric and silently destroyed `F3889` / `F3891`).

In [ ]:
raw_df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Loaded dataset shape: {raw_df.shape}")

df = raw_df.copy()
if ID_COL in df.columns:
    df = df.drop(columns=[ID_COL])

y = df[TARGET_COL].astype(int)
raw_features = df.drop(columns=[TARGET_COL], errors="ignore")
raw_features = raw_features.replace([np.inf, -np.inf], np.nan)
raw_features = raw_features.replace(list(PLACEHOLDER_VALUES), np.nan)

print("Feature matrix:", raw_features.shape)
print("Target counts:\n", y.value_counts().sort_index().to_string())
print("Target base rate:", float(y.mean()))
print("Duplicate rows:", int(pd.concat([raw_features, y], axis=1).duplicated().sum()))
print("Bank features present:", [c for c in BANK_FEATURES if c in raw_features.columns])
print("Bank features missing:", [c for c in BANK_FEATURES if c not in raw_features.columns])
print("Accused columns present:", [c for c in PREVIOUSLY_ACCUSED if c in raw_features.columns])

## 2. Leakage retest — methodology

Phase 1 step 4 used:

`max fraud rate > 0.99` **or** `min fraud rate < 0.001` → “HIGHLY LEAKY”

The positive class is **0.89%**. Any category with a handful of legitimate accounts and zero fraud then has `min fraud rate = 0` and gets banned. That is how `F3886`, `F3889`, `F3891`, and `F3892` were marked leaky.

This cell uses rules that can actually support an argument:

1. **Numeric proxy:** \|Pearson corr\| ≥ 0.50 **and** \|Cohen's d\| ≥ 2.0, or near-copy of the label (same support ±2).
2. **Categorical purity:** at least one category with **n ≥ 20** and **fraud rate ≥ 0.99**, or a category that contains **every** fraud row while the complement is all-negative with n ≥ 20.
3. **Old heuristic (reproduced)** so we can show the false positives in a table.
4. **Holdout ablation:** one stratified 80/20 XGBoost run with vs without each accused column, so the PR-AUC lift is a number, not a story.

`Description.xlsx` is not consulted. Labels below come only from the data.

In [ ]:
def series_kind(s: pd.Series) -> str:
    if pd.api.types.is_numeric_dtype(s) and s.nunique(dropna=True) > 12:
        return "numeric"
    if pd.api.types.is_numeric_dtype(s) and s.nunique(dropna=True) <= 12:
        return "numeric_low_card"
    return "categorical"


def cohen_d(col: pd.Series, target: pd.Series) -> float:
    g0 = col[target == 0].dropna().astype(float)
    g1 = col[target == 1].dropna().astype(float)
    if len(g0) < 2 or len(g1) < 2:
        return np.nan
    pooled = np.sqrt((g0.var(ddof=1) + g1.var(ddof=1)) / 2.0)
    if pooled == 0 or np.isnan(pooled):
        return np.nan
    return float((g1.mean() - g0.mean()) / pooled)


def old_heuristic_flag(fraud_rates: list[float]) -> bool:
    if not fraud_rates:
        return False
    return (max(fraud_rates) > 0.99) or (min(fraud_rates) < 0.001)


def category_purity_table(col: pd.Series, target: pd.Series) -> pd.DataFrame:
    filled = col.astype("string").fillna("__NA__")
    rows = []
    for val, n in filled.value_counts().items():
        mask = filled == val
        fraud_n = int(target[mask].sum())
        rows.append({
            "category": val,
            "n": int(n),
            "fraud_n": fraud_n,
            "fraud_rate": float(target[mask].mean()),
            "support_ok": int(n) >= PURITY_MIN_SUPPORT,
            "pure_positive": (int(n) >= PURITY_MIN_SUPPORT) and (fraud_n / n >= PURITY_MIN_RATE),
        })
    return pd.DataFrame(rows).sort_values(["pure_positive", "fraud_rate", "n"], ascending=False)


def audit_column(col_name: str, col: pd.Series, target: pd.Series) -> dict:
    kind = series_kind(col)
    result = {
        "feature": col_name,
        "kind": kind,
        "nunique": int(col.nunique(dropna=False)),
        "missing_rate": float(col.isna().mean()),
        "old_heuristic_leaky": False,
        "retest_leaky": False,
        "reason": "",
        "abs_corr": np.nan,
        "cohen_d": np.nan,
        "max_fraud_rate": np.nan,
        "min_fraud_rate": np.nan,
        "pure_positive_categories": 0,
    }

    if kind in {"numeric", "numeric_low_card"}:
        numeric = pd.to_numeric(col, errors="coerce")
        result["abs_corr"] = float(numeric.corr(target)) if numeric.notna().any() else np.nan
        result["cohen_d"] = cohen_d(numeric, target)
        # also treat low-card numeric as categories for the old heuristic
        rates = []
        for val, n in numeric.fillna(-999999).value_counts().items():
            mask = numeric.fillna(-999999) == val
            rates.append(float(target[mask].mean()))
        result["max_fraud_rate"] = max(rates) if rates else np.nan
        result["min_fraud_rate"] = min(rates) if rates else np.nan
        result["old_heuristic_leaky"] = old_heuristic_flag(rates)
        near_copy = (
            numeric.nunique(dropna=True) == 2
            and abs(int((numeric == 1).sum()) - int((target == 1).sum())) <= 2
            and abs(result["abs_corr"]) >= 0.90
        )
        proxy = (
            (abs(result["abs_corr"]) >= PROXY_MIN_ABS_CORR and abs(result["cohen_d"]) >= PROXY_MIN_ABS_D)
            if pd.notna(result["abs_corr"]) and pd.notna(result["cohen_d"])
            else False
        )
        result["retest_leaky"] = bool(near_copy or proxy)
        if near_copy:
            result["reason"] = "numeric near-copy of the label"
        elif proxy:
            result["reason"] = f"|corr|={result['abs_corr']:.3f}, |d|={abs(result['cohen_d']):.2f}"
        else:
            result["reason"] = "numeric; no proxy signature"
    else:
        purity = category_purity_table(col, target)
        result["max_fraud_rate"] = float(purity["fraud_rate"].max())
        result["min_fraud_rate"] = float(purity["fraud_rate"].min())
        result["pure_positive_categories"] = int(purity["pure_positive"].sum())
        result["old_heuristic_leaky"] = old_heuristic_flag(purity["fraud_rate"].tolist())
        result["retest_leaky"] = result["pure_positive_categories"] > 0
        if result["retest_leaky"]:
            hot = purity.loc[purity["pure_positive"], "category"].tolist()
            result["reason"] = f"pure-positive categories with n>={PURITY_MIN_SUPPORT}: {hot}"
        else:
            result["reason"] = (
                "no category with support "
                f">={PURITY_MIN_SUPPORT} and fraud rate >={PURITY_MIN_RATE}; "
                "zero-fraud cells are expected at a 0.89% base rate"
            )
        result["_purity"] = purity
    return result


audit_rows = []
purity_book = {}
for col_name in PREVIOUSLY_ACCUSED:
    payload = audit_column(col_name, raw_features[col_name], y)
    purity_book[col_name] = payload.pop("_purity", None)
    audit_rows.append(payload)

audit_df = pd.DataFrame(audit_rows)
print("=== LEAKAGE RETEST vs OLD HEURISTIC ===")
display(audit_df)

print("\n=== Category purity tables (factual backup) ===")
for col_name in PREVIOUSLY_ACCUSED:
    if purity_book.get(col_name) is None:
        continue
    print(f"\n{col_name}")
    display(purity_book[col_name])

### 2b. Holdout ablation — does the accused column actually move PR-AUC?

Each accused column is added on top of a **non-leaky compact numeric table** (bank numerics + row stats, no MI, no other accused columns). A single stratified 80/20 XGBoost run reports PR-AUC with and without that column. Large lifts that collapse after a proper audit are the signature of leakage; tiny lifts are ordinary weak signal.

In [ ]:
def detect_placeholder_values(frame: pd.DataFrame, abs_threshold: float, min_frac: float) -> dict[str, float]:
    placeholder_map: dict[str, float] = {}
    for col in frame.columns:
        series = pd.to_numeric(frame[col], errors="coerce").dropna()
        if series.empty:
            continue
        extreme = series[series.abs() >= abs_threshold]
        if extreme.empty:
            continue
        counts = extreme.value_counts()
        candidate = counts.index[0]
        if counts.iloc[0] / len(series) >= min_frac:
            placeholder_map[col] = float(candidate)
    return placeholder_map


def build_row_stats(frame: pd.DataFrame) -> pd.DataFrame:
    values = frame.to_numpy(dtype=float)
    mask = ~np.isnan(values)
    non_missing = mask.sum(axis=1)
    total = max(values.shape[1], 1)
    with np.errstate(all="ignore"):
        mean = np.nanmean(values, axis=1)
        std = np.nanstd(values, axis=1)
        min_val = np.nanmin(values, axis=1)
        max_val = np.nanmax(values, axis=1)
        median = np.nanmedian(values, axis=1)
        q25 = np.nanpercentile(values, 25, axis=1)
        q75 = np.nanpercentile(values, 75, axis=1)
        abs_mean = np.nanmean(np.abs(values), axis=1)
    iqr = q75 - q25
    return pd.DataFrame({
        "row_non_missing_count": non_missing,
        "row_missing_rate": 1.0 - (non_missing / total),
        "row_zero_rate": np.where(non_missing > 0, (values == 0).sum(axis=1) / non_missing, 0),
        "row_positive_rate": np.where(non_missing > 0, (values > 0).sum(axis=1) / non_missing, 0),
        "row_negative_rate": np.where(non_missing > 0, (values < 0).sum(axis=1) / non_missing, 0),
        "row_mean": mean,
        "row_std": std,
        "row_min": min_val,
        "row_max": max_val,
        "row_median": median,
        "row_q25": q25,
        "row_q75": q75,
        "row_iqr": iqr,
        "row_abs_mean": abs_mean,
    }, index=frame.index)


def encode_one_column(train: pd.Series, valid: pd.Series, y_train: pd.Series, name: str) -> tuple[pd.Series, pd.Series]:
    train_s = train.astype("string").fillna("MISSING")
    valid_s = valid.astype("string").fillna("MISSING")
    if pd.api.types.is_numeric_dtype(train) and train.nunique(dropna=True) <= 12:
        return pd.to_numeric(train, errors="coerce"), pd.to_numeric(valid, errors="coerce")
    card = train_s.nunique(dropna=False)
    if card <= LOW_CARD_MAX_UNIQUE:
        mapping = {v: i for i, v in enumerate(sorted(train_s.unique().tolist()))}
        return (
            train_s.map(mapping).fillna(-1).astype(float).rename(name),
            valid_s.map(mapping).fillna(-1).astype(float).rename(name),
        )
    global_mean = float(y_train.mean())
    stats = pd.DataFrame({"category": train_s, "target": y_train.to_numpy()}).groupby("category")["target"].agg(["mean", "count"])
    smooth = (stats["count"] * stats["mean"] + TARGET_ENCODING_SMOOTHING * global_mean) / (stats["count"] + TARGET_ENCODING_SMOOTHING)
    return (
        train_s.map(smooth).fillna(global_mean).astype(float).rename(name),
        valid_s.map(smooth).fillna(global_mean).astype(float).rename(name),
    )


numeric_all = raw_features.apply(pd.to_numeric, errors="coerce")
placeholder_map = detect_placeholder_values(numeric_all, LARGE_ABS_THRESHOLD, PLACEHOLDER_MIN_FRAC)
for col, value in placeholder_map.items():
    numeric_all[col] = numeric_all[col].replace(value, np.nan)

row_stats_all = build_row_stats(numeric_all)
bank_numeric = [c for c in BANK_FEATURES if c in numeric_all.columns and numeric_all[c].notna().any()]
ablation_base = pd.concat([numeric_all[bank_numeric], row_stats_all], axis=1)
ablation_base = ablation_base.loc[:, ablation_base.isna().mean() < 1.0]

print("Ablation base shape:", ablation_base.shape)
print("Placeholder columns detected:", len(placeholder_map))


def pr_auc_with_optional_column(extra_col: str | None, n_estimators: int = 200) -> float:
    if xgb is None:
        return np.nan
    X_tr, X_va, y_tr, y_va, raw_tr, raw_va = train_test_split(
        ablation_base, y, raw_features, test_size=0.2, stratify=y, random_state=RANDOM_STATE,
    )
    if extra_col is not None:
        extra_tr, extra_va = encode_one_column(raw_tr[extra_col], raw_va[extra_col], y_tr, extra_col)
        X_tr = X_tr.copy()
        X_va = X_va.copy()
        X_tr[extra_col] = extra_tr.to_numpy()
        X_va[extra_col] = extra_va.to_numpy()
    imputer = SimpleImputer(strategy="median")
    X_tr_i = imputer.fit_transform(X_tr)
    X_va_i = imputer.transform(X_va)
    spw = float((y_tr == 0).sum() / max((y_tr == 1).sum(), 1))
    model = xgb.XGBClassifier(
        n_estimators=n_estimators,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="aucpr",
        scale_pos_weight=spw,
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    model.fit(X_tr_i, y_tr)
    probs = model.predict_proba(X_va_i)[:, 1]
    return float(average_precision_score(y_va, probs))


print("\nFitting holdout ablation (one 80/20 split, 200 trees, no SMOTE)...")
base_pr = pr_auc_with_optional_column(None)
ablation_rows = [{"feature": "(base: bank numeric + row stats)", "pr_auc": base_pr, "delta_vs_base": 0.0}]
for col_name in PREVIOUSLY_ACCUSED:
    pr = pr_auc_with_optional_column(col_name)
    ablation_rows.append({"feature": f"+ {col_name}", "pr_auc": pr, "delta_vs_base": pr - base_pr})

ablation_df = pd.DataFrame(ablation_rows)
print("\n=== HOLDOUT ABLATION (PR-AUC) ===")
display(ablation_df)

### 2c. Leakage decision (locked for the rest of this notebook)

The cells above are the evidence. The locked policy:

- **Drop `F3912`.** Binary almost equal to `F3924` (corr ≈ 0.97). Target proxy.
- **Drop `F2230`.** `Sep25` / `Nov25` / `Dec25` are 100% fraud with support 48 / 23 / 10; `Oct25` is 0% fraud with support 9001. Post-hoc cohort stamp.
- **Keep `F3886`.** Account product type. Zero-fraud MSME cells are small-n, not purity.
- **Salvage `F3889` as `F3889_comp_lag`.** Values are recency buckets (`L7D` … `G365D`), which is what PS2 asked for, not a date and not a leak.
- **Keep `F3891` as categorical occupation.** PS2’s “prior audit binary” does not match the observed values (`selfemployed`, `salaried`, `student`, …). Dropping it was the mistake; inventing a recidivism bit would be a second mistake.
- **Keep `F3892`.** Gender. The `O` cell has n=61 and 0 fraud — expected, not a leak.

In [ ]:
CONFIRMED_LEAKY = audit_df.loc[audit_df["retest_leaky"], "feature"].tolist()
# Hard lock matches the evidence we just printed. If the retest ever disagrees, fail loudly.
EXPECTED_LEAKY = ["F3912", "F2230"]
if sorted(CONFIRMED_LEAKY) != sorted(EXPECTED_LEAKY):
    print("WARNING: retest set", CONFIRMED_LEAKY, "differs from expected", EXPECTED_LEAKY)
    print("Proceeding with the union so we never accidentally train on a confirmed proxy.")
    CONFIRMED_LEAKY = sorted(set(CONFIRMED_LEAKY) | set(EXPECTED_LEAKY))

SALVAGE_LAG = ["F3889"]
KEEP_CATEGORICAL_BANK = ["F3891"]

print("Confirmed leaky (excluded from modeling):", CONFIRMED_LEAKY)
print("Salvaged as numeric lag:", SALVAGE_LAG)
print("Bank categoricals kept:", KEEP_CATEGORICAL_BANK)

audit_path = OUT_DIR / "leakage_retest.json"
audit_payload = {
    "confirmed_leaky": CONFIRMED_LEAKY,
    "old_heuristic": audit_df[["feature", "old_heuristic_leaky", "retest_leaky", "reason"]].to_dict(orient="records"),
    "holdout_ablation": ablation_df.to_dict(orient="records"),
    "description_xlsx_used": False,
}
audit_path.write_text(json.dumps(audit_payload, indent=2), encoding="utf-8")
print("Wrote", audit_path)

## 3. Clean matrix, salvage `F3889`, classify dtypes

Placeholders and numeric coercion are copied from Phase 1. Confirmed leaks are dropped **after** the audit. Temporal detection runs only on remaining object columns so `F2230` cannot sneak in as a “date”.

In [ ]:
work = raw_features.drop(columns=[c for c in CONFIRMED_LEAKY if c in raw_features.columns], errors="ignore")

if "F3889" in work.columns:
    work["F3889_comp_lag"] = work["F3889"].astype("string").map(F3889_LAG_MAP).astype(float)
    print("F3889_comp_lag value counts:")
    print(work["F3889_comp_lag"].value_counts(dropna=False).to_string())
    work = work.drop(columns=["F3889"])

object_cols = work.select_dtypes(include=["object", "string", "category"]).columns.tolist()


def identify_temporal_columns(frame: pd.DataFrame, min_frac: float = TEMPORAL_PARSE_MIN_FRAC, sample_size: int = 5000) -> list[str]:
    temporal_cols: list[str] = []
    for col in frame.columns:
        series = frame[col].dropna().astype(str)
        if series.empty:
            continue
        if len(series) > sample_size:
            series = series.sample(sample_size, random_state=RANDOM_STATE)
        parsed = pd.to_datetime(series, errors="coerce")
        if parsed.notna().mean() >= min_frac and parsed.nunique(dropna=True) > 1:
            temporal_cols.append(col)
    return temporal_cols


temporal_cols = identify_temporal_columns(work[object_cols]) if object_cols else []
categorical_cols = [c for c in object_cols if c not in temporal_cols]

numeric_base = work.apply(pd.to_numeric, errors="coerce")
for col, value in placeholder_map.items():
    if col in numeric_base.columns:
        numeric_base[col] = numeric_base[col].replace(value, np.nan)

row_stats = build_row_stats(numeric_base)

print("Numeric base:", numeric_base.shape)
print("Object columns:", object_cols)
print("Temporal columns:", temporal_cols)
print("Categorical columns:", categorical_cols)
print("Target base rate:", float(y.mean()))

## 4. Compact feature table (PS2 tracks)

Copied from Phase 1 `build_step2_table`, with two changes:

- MI and missingness-gap selection are **train-fold only** inside CV.
- Gap flags only for columns whose class-conditional missingness gap is ≥ 20% (PS2), still capped at top 25.

PCA / KMeans / SMOTE / `elapsed_days` are not used.

In [ ]:
def select_compact_columns(numeric_frame: pd.DataFrame, target: pd.Series) -> dict:
    bank_features = [c for c in BANK_FEATURES if c in numeric_frame.columns and numeric_frame[c].notna().any()]
    if "F3889_comp_lag" in numeric_frame.columns and numeric_frame["F3889_comp_lag"].notna().any():
        if "F3889_comp_lag" not in bank_features:
            bank_features.append("F3889_comp_lag")

    nunique = numeric_frame.nunique(dropna=True)
    miss = numeric_frame.isna().mean()
    usable = [c for c in numeric_frame.columns if nunique.get(c, 0) > 1 and miss.get(c, 1) < 0.95]
    mi_candidates = numeric_frame[usable]
    if mi_candidates.shape[1] > MI_CANDIDATE_CAP:
        variances = mi_candidates.var(skipna=True).sort_values(ascending=False)
        mi_candidates = mi_candidates[variances.head(MI_CANDIDATE_CAP).index.tolist()]

    top_mi_cols: list[str] = []
    if mi_candidates.shape[1] > 0:
        mi_values = SimpleImputer(strategy="median").fit_transform(mi_candidates)
        mi_scores = mutual_info_classif(mi_values, target, random_state=RANDOM_STATE)
        top_mi_cols = pd.Series(mi_scores, index=mi_candidates.columns).sort_values(ascending=False).head(TOP_MI).index.tolist()

    missing_gap = (
        numeric_frame.loc[target == 1].isna().mean()
        - numeric_frame.loc[target == 0].isna().mean()
    ).abs().sort_values(ascending=False)
    top_gap_cols = missing_gap[missing_gap >= GAP_MIN].head(TOP_GAP).index.tolist()
    if not top_gap_cols:
        top_gap_cols = missing_gap.head(TOP_GAP).index.tolist()

    selected: list[str] = []
    for col in bank_features + top_mi_cols + top_gap_cols:
        if col not in selected and col in numeric_frame.columns:
            selected.append(col)
    return {"bank_features": bank_features, "top_mi_cols": top_mi_cols, "top_gap_cols": top_gap_cols, "selected_cols": selected}


def apply_compact(numeric_frame: pd.DataFrame, row_stats_frame: pd.DataFrame, spec: dict) -> pd.DataFrame:
    gap_cols = [c for c in spec["top_gap_cols"] if c in numeric_frame.columns]
    missing_flags = numeric_frame[gap_cols].isna().astype(int).add_prefix("miss_") if gap_cols else pd.DataFrame(index=numeric_frame.index)
    compact = pd.concat([numeric_frame[spec["selected_cols"]], row_stats_frame, missing_flags], axis=1)
    return compact.loc[:, compact.isna().mean() < 1.0]


def encode_categorical_fold(train_raw: pd.DataFrame, valid_raw: pd.DataFrame, train_target: pd.Series, cols: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_parts, valid_parts = [], []
    for col in cols:
        if col not in train_raw.columns:
            continue
        train_series = train_raw[col].astype("string").fillna("MISSING")
        valid_series = valid_raw[col].astype("string").fillna("MISSING")
        cardinality = train_series.nunique(dropna=False)
        if cardinality <= LOW_CARD_MAX_UNIQUE:
            mapping = {value: idx for idx, value in enumerate(sorted(train_series.unique().tolist()))}
            train_encoded = train_series.map(mapping).fillna(-1).astype(float)
            valid_encoded = valid_series.map(mapping).fillna(-1).astype(float)
            name = f"{col}_ord"
        else:
            global_mean = float(train_target.mean())
            stats = pd.DataFrame({"category": train_series, "target": train_target.to_numpy()}).groupby("category")["target"].agg(["mean", "count"])
            smooth = (stats["count"] * stats["mean"] + TARGET_ENCODING_SMOOTHING * global_mean) / (stats["count"] + TARGET_ENCODING_SMOOTHING)
            train_encoded = train_series.map(smooth).fillna(global_mean).astype(float)
            valid_encoded = valid_series.map(smooth).fillna(global_mean).astype(float)
            name = f"{col}_te"
        train_parts.append(pd.Series(train_encoded.to_numpy(), index=train_raw.index, name=name))
        valid_parts.append(pd.Series(valid_encoded.to_numpy(), index=valid_raw.index, name=name))
    if train_parts:
        return pd.concat(train_parts, axis=1), pd.concat(valid_parts, axis=1)
    return pd.DataFrame(index=train_raw.index), pd.DataFrame(index=valid_raw.index)


def build_temporal_fold_features(train_raw: pd.DataFrame, valid_raw: pd.DataFrame, cols: list[str], drop_elapsed_days: bool = True) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_parts, valid_parts = [], []
    for col in cols:
        train_parsed = pd.to_datetime(train_raw[col], errors="coerce")
        valid_parsed = pd.to_datetime(valid_raw[col], errors="coerce")

        def make_temporal_frame(parsed: pd.Series) -> pd.DataFrame:
            dow = parsed.dt.dayofweek.astype(float)
            month = parsed.dt.month.astype(float)
            features = {
                f"{col}_dow_sin": np.sin(2 * np.pi * dow / 7.0),
                f"{col}_dow_cos": np.cos(2 * np.pi * dow / 7.0),
                f"{col}_month_sin": np.sin(2 * np.pi * (month - 1.0) / 12.0),
                f"{col}_month_cos": np.cos(2 * np.pi * (month - 1.0) / 12.0),
            }
            if not drop_elapsed_days:
                base_date = train_parsed.min()
                if pd.isna(base_date):
                    base_date = pd.Timestamp("1970-01-01")
                features[f"{col}_elapsed_days"] = (parsed - base_date).dt.total_seconds() / 86400.0
            return pd.DataFrame(features, index=parsed.index)

        train_parts.append(make_temporal_frame(train_parsed))
        valid_parts.append(make_temporal_frame(valid_parsed))
    if train_parts:
        return pd.concat(train_parts, axis=1), pd.concat(valid_parts, axis=1)
    return pd.DataFrame(index=train_raw.index), pd.DataFrame(index=valid_raw.index)


def add_anomaly_feature(train_frame: pd.DataFrame, valid_frame: pd.DataFrame, train_target: pd.Series) -> tuple[pd.DataFrame, pd.DataFrame]:
    imputer = SimpleImputer(strategy="median")
    train_imp = imputer.fit_transform(train_frame) if train_frame.shape[1] else np.zeros((len(train_frame), 1))
    valid_imp = imputer.transform(valid_frame) if valid_frame.shape[1] else np.zeros((len(valid_frame), 1))
    scaler = RobustScaler()
    train_scaled = scaler.fit_transform(train_imp)
    valid_scaled = scaler.transform(valid_imp)
    normal_mask = (train_target == 0).to_numpy()
    if normal_mask.sum() < 10:
        normal_mask = np.ones(len(train_target), dtype=bool)
    iso = IsolationForest(n_estimators=ANOMALY_N_ESTIMATORS, random_state=RANDOM_STATE, n_jobs=-1)
    iso.fit(train_scaled[normal_mask])
    train_aug = train_frame.copy()
    valid_aug = valid_frame.copy()
    train_aug["iso_anomaly_score"] = -iso.score_samples(train_scaled)
    valid_aug["iso_anomaly_score"] = -iso.score_samples(valid_scaled)
    return train_aug, valid_aug


def augment_fold(base_train, base_valid, raw_train, raw_valid, train_target):
    cat_tr, cat_va = encode_categorical_fold(raw_train, raw_valid, train_target, categorical_cols)
    temp_tr, temp_va = build_temporal_fold_features(raw_train, raw_valid, temporal_cols, drop_elapsed_days=True)
    X_tr = pd.concat([base_train, cat_tr, temp_tr], axis=1)
    X_va = pd.concat([base_valid, cat_va, temp_va], axis=1)
    return add_anomaly_feature(X_tr, X_va, train_target)


print("Helper functions ready. Compact selection happens inside each CV fold.")

## 5. Ensemble, calibration, cost threshold

Copied from Phase 1 steps 5–6, with SMOTE removed (PS2: *completely eliminates SMOTE oversampling*). Blend is `0.6 XGBoost + 0.4 LightGBM`. Thresholds are chosen on the **validation fold** the same way the original notebooks did; that is slightly optimistic for F1/cost and is called out in the summary. Ranking metrics (PR-AUC, ROC-AUC) do not use the threshold.

In [ ]:
def build_xgb(scale_pos_weight: float, n_estimators: int = 500):
    return xgb.XGBClassifier(
        n_estimators=n_estimators,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="aucpr",
        scale_pos_weight=scale_pos_weight,
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )


def build_lgb(scale_pos_weight: float, n_estimators: int = 500):
    return lgb.LGBMClassifier(
        n_estimators=n_estimators,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary",
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1,
    )


def calibrate_probabilities(y_tr, probs_tr, probs_va):
    calibrator = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1)
    calibrator.fit(probs_tr, y_tr)
    return calibrator.predict(probs_va), calibrator


def compute_cost_metrics(y_true, y_pred, cost_fn_ratio=COST_FN_RATIO, cost_fp=COST_FP_BASE):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    total_cost = (fn * cost_fn_ratio * cost_fp) + (fp * cost_fp)
    return {"tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn), "total_cost": float(total_cost)}


def find_cost_optimal_threshold(y_true, probs, cost_fn_ratio=COST_FN_RATIO, cost_fp=COST_FP_BASE):
    unique_probs = np.unique(probs)
    min_cost, optimal_thr = np.inf, 0.5
    for thr in unique_probs:
        cost = compute_cost_metrics(y_true, (probs >= thr).astype(int), cost_fn_ratio, cost_fp)["total_cost"]
        if cost < min_cost:
            min_cost, optimal_thr = cost, thr
    return optimal_thr, min_cost


def find_f1_optimal_threshold(y_true, probs):
    prec, rec, thrs = precision_recall_curve(y_true, probs)
    if len(thrs) == 0:
        return 0.5
    f1s = (2 * prec[:-1] * rec[:-1]) / (prec[:-1] + rec[:-1] + 1e-12)
    return float(thrs[int(np.argmax(f1s))])


def fit_blend(X_tr, y_tr, X_va):
    imputer = SimpleImputer(strategy="median")
    X_tr_i = imputer.fit_transform(X_tr)
    X_va_i = imputer.transform(X_va)
    spw = float((y_tr == 0).sum() / max((y_tr == 1).sum(), 1))
    xgb_fold = build_xgb(spw).fit(X_tr_i, y_tr)
    lgb_fold = build_lgb(spw).fit(X_tr_i, y_tr)
    w0, w1 = BLEND_WEIGHTS
    probs_tr = w0 * xgb_fold.predict_proba(X_tr_i)[:, 1] + w1 * lgb_fold.predict_proba(X_tr_i)[:, 1]
    probs_va = w0 * xgb_fold.predict_proba(X_va_i)[:, 1] + w1 * lgb_fold.predict_proba(X_va_i)[:, 1]
    return probs_tr, probs_va


def evaluate_split(X_tr, X_va, y_tr, y_va, raw_tr, raw_va, fold_label: str) -> dict:
    spec = select_compact_columns(X_tr, y_tr)
    base_tr = apply_compact(X_tr, row_stats.loc[X_tr.index], spec)
    base_va = apply_compact(X_va, row_stats.loc[X_va.index], spec)
    feat_tr, feat_va = augment_fold(base_tr, base_va, raw_tr, raw_va, y_tr)
    probs_tr, probs_va = fit_blend(feat_tr, y_tr, feat_va)
    probs_va_cal, _ = calibrate_probabilities(y_tr.to_numpy(), probs_tr, probs_va)
    cost_thr, _ = find_cost_optimal_threshold(y_va.to_numpy(), probs_va_cal)
    f1_thr = find_f1_optimal_threshold(y_va.to_numpy(), probs_va_cal)
    pred_cost = (probs_va_cal >= cost_thr).astype(int)
    pred_f1 = (probs_va_cal >= f1_thr).astype(int)
    cost_m = compute_cost_metrics(y_va.to_numpy(), pred_cost)
    f1_m = compute_cost_metrics(y_va.to_numpy(), pred_f1)
    row = {
        "fold": fold_label,
        "n_features": int(feat_tr.shape[1]),
        "pr_auc": float(average_precision_score(y_va, probs_va_cal)),
        "roc_auc": float(roc_auc_score(y_va, probs_va_cal)),
        "macro_f1": float(f1_score(y_va, pred_cost, average="macro")),
        "minority_f1": float(f1_score(y_va, pred_cost, pos_label=1)),
        "balanced_acc": float(balanced_accuracy_score(y_va, pred_cost)),
        "cost_thr": float(cost_thr),
        "f1_thr": float(f1_thr),
        "total_cost_cost_opt": cost_m["total_cost"],
        "total_cost_f1_opt": f1_m["total_cost"],
        "fn_cost_opt": cost_m["fn"],
        "fp_cost_opt": cost_m["fp"],
    }
    savings = 0.0 if f1_m["total_cost"] == 0 else 100.0 * (f1_m["total_cost"] - cost_m["total_cost"]) / f1_m["total_cost"]
    row["cost_savings_pct"] = float(savings)
    print(
        f"{fold_label}: feats={row['n_features']} PR-AUC={row['pr_auc']:.4f} "
        f"MacroF1={row['macro_f1']:.4f} MinF1={row['minority_f1']:.4f} "
        f"Cost={row['total_cost_cost_opt']:.1f} (F1-opt {row['total_cost_f1_opt']:.1f})"
    )
    return row


print("Trainers ready. No SMOTE. scale_pos_weight = n_neg / n_pos on each train fold.")

## 6. Stratified 5-fold CV (interpolation)

Matches the PS2 “Audited Step 4 Clean Pipeline” protocol, minus the leaks the retest confirmed and minus SMOTE.

In [ ]:
if xgb is None or lgb is None:
    raise RuntimeError("xgboost and lightgbm are required for the stable baseline.")

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
random_rows = []
for fold, (train_idx, val_idx) in enumerate(skf.split(numeric_base, y), start=1):
    random_rows.append(evaluate_split(
        numeric_base.iloc[train_idx],
        numeric_base.iloc[val_idx],
        y.iloc[train_idx],
        y.iloc[val_idx],
        work.iloc[train_idx],
        work.iloc[val_idx],
        f"random-{fold}",
    ))

random_df = pd.DataFrame(random_rows)
print("\n=== Stratified 5-fold (mean ± std) ===")
summary_cols = ["pr_auc", "roc_auc", "macro_f1", "minority_f1", "balanced_acc", "total_cost_cost_opt", "total_cost_f1_opt", "cost_savings_pct", "n_features"]
display(random_df[summary_cols].agg(["mean", "std"]).T)
display(random_df)

## 7. Chronological validation (extrapolation)

Ordered by `F3888` (account registration). Copied from step 5 / PS2 Table 4. This is the honest deployment estimate.

In [ ]:
chrono_df = pd.DataFrame()
if temporal_cols:
    order_col = temporal_cols[0]
    parsed_dates = pd.to_datetime(work[order_col], errors="coerce")
    order = parsed_dates.argsort(kind="mergesort")
    n = len(order)
    # 5 expanding chronological windows: train on earliest (k+1)/6, test on the next 1/6.
    chrono_rows = []
    for fold in range(N_SPLITS):
        train_end = int(n * (fold + 1) / (N_SPLITS + 1))
        test_end = int(n * (fold + 2) / (N_SPLITS + 1))
        tr_idx = order[:train_end]
        va_idx = order[train_end:test_end]
        y_va = y.iloc[va_idx]
        if y_va.nunique() < 2 or int(y_va.sum()) == 0:
            print(f"chrono-{fold}: skipped (no positive labels in the test window)")
            continue
        chrono_rows.append(evaluate_split(
            numeric_base.iloc[tr_idx],
            numeric_base.iloc[va_idx],
            y.iloc[tr_idx],
            y_va,
            work.iloc[tr_idx],
            work.iloc[va_idx],
            f"chrono-{fold}",
        ))
    chrono_df = pd.DataFrame(chrono_rows)
    print("\n=== Chronological (mean ± std) ===")
    display(chrono_df[summary_cols].agg(["mean", "std"]).T)
    display(chrono_df)
else:
    print("No temporal column found after dropping confirmed leaks; chronological split skipped.")

## 8. Locked baseline vs PS2 golden-truth numbers

PS2 Table 5 quoted PR-AUC **0.8677** / Macro F1 **0.9147**. Those numbers were produced by the leaky-feature-dropped + SMOTE pipeline on **step 3’s globally fitted table**, not by the protocol this notebook actually implements. We report our measured numbers next to that quote so the argument stays factual.

In [ ]:
ps2_quote = pd.DataFrame({
    "metric": ["PR-AUC", "ROC-AUC", "Macro F1", "Minority F1", "Balanced Acc"],
    "ps2_table5_quoted": [0.867695, 0.984044, 0.914667, 0.830833, 0.900492],
    "this_notebook_random_mean": [
        random_df["pr_auc"].mean(),
        random_df["roc_auc"].mean(),
        random_df["macro_f1"].mean(),
        random_df["minority_f1"].mean(),
        random_df["balanced_acc"].mean(),
    ],
})
if not chrono_df.empty:
    ps2_quote["this_notebook_chrono_mean"] = [
        chrono_df["pr_auc"].mean(),
        chrono_df["roc_auc"].mean(),
        chrono_df["macro_f1"].mean(),
        chrono_df["minority_f1"].mean(),
        chrono_df["balanced_acc"].mean(),
    ]

print("=== PS2 quote vs this notebook ===")
display(ps2_quote)

print("\n=== Leakage policy (from cells 2–2c) ===")
display(audit_df[["feature", "old_heuristic_leaky", "retest_leaky", "reason"]])
print("\nHoldout ablation:")
display(ablation_df)

print("\nDescription.xlsx used:", False)
print("SMOTE used:", False)
print("elapsed_days used:", False)
print("Confirmed leaks excluded:", CONFIRMED_LEAKY)

metrics_path = OUT_DIR / "baseline_metrics.json"
metrics_path.write_text(json.dumps({
    "confirmed_leaky": CONFIRMED_LEAKY,
    "random_cv": random_df.to_dict(orient="records"),
    "chrono_cv": chrono_df.to_dict(orient="records") if not chrono_df.empty else [],
    "ps2_comparison": ps2_quote.to_dict(orient="records"),
    "cost_fn_ratio": COST_FN_RATIO,
}, indent=2, default=float), encoding="utf-8")
print("Wrote", metrics_path)

## 9. What is deliberately *not* in this baseline

- **`Description.xlsx`** — attached as a new data dictionary. Unused here. Phase 2 fine-tuning will map `Variable Name` / `Description` / `Bank_Finalized_Variables`.
- **Ego-graph / Redis / FastAPI / TreeSHAP-PCHIP demo** — specified in PS2 sections 3 and 7 and in `phase2_roadmap.md`. Not implemented in any Phase 1 notebook; they are expansion work, not silently faked.
- **Invented `F3891_prior_audit` bit** — the column is occupation, not an investigation flag.

This notebook is the Phase 1 locked baseline. Phase 2 starts from these artifacts (`leakage_retest.json`, `baseline_metrics.json`) rather than from the older step notebooks.